In [148]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kushleshkumar/cornell-car-rental-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\judep\.cache\kagglehub\datasets\kushleshkumar\cornell-car-rental-dataset\versions\2


In [149]:
#load dataset into pandas dataframe
import pandas as pd

df = pd.read_csv('carrentaldata.csv')

In [150]:
#show first 5 rows of the dataframe
df.head()

,fuelType,rating,renterTripsTaken,reviewCount,location.city,location.country,location.latitude,location.longitude,location.state,owner.id,rate.daily,vehicle.make,vehicle.model,vehicle.type,vehicle.year
0,ELECTRIC,5.00,13,12,Seattle,US,47.449107,-122.308841,WA,12847615,135,Tesla,Model X,suv,2019
1,ELECTRIC,5.00,2,1,Tijeras,US,35.111060,-106.276551,NM,15621242,190,Tesla,Model X,suv,2018
2,HYBRID,4.92,28,24,Albuquerque,US,35.127163,-106.566681,NM,10199256,35,Toyota,Prius,car,2012
3,GASOLINE,5.00,21,20,Albuquerque,US,35.149726,-106.711425,NM,9365496,75,Ford,Mustang,car,2018
4,GASOLINE,5.00,3,1,Albuquerque,US,35.208659,-106.601008,NM,3553565,47,Chrysler,Sebring,car,2010


In [151]:
#check null values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5851 entries, 0 to 5850
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   fuelType            5776 non-null   str    
 1   rating              5350 non-null   float64
 2   renterTripsTaken    5851 non-null   int64  
 3   reviewCount         5851 non-null   int64  
 4   location.city       5851 non-null   str    
 5   location.country    5851 non-null   str    
 6   location.latitude   5851 non-null   float64
 7   location.longitude  5851 non-null   float64
 8   location.state      5851 non-null   str    
 9   owner.id            5851 non-null   int64  
 10  rate.daily          5851 non-null   int64  
 11  vehicle.make        5851 non-null   str    
 12  vehicle.model       5851 non-null   str    
 13  vehicle.type        5851 non-null   str    
 14  vehicle.year        5851 non-null   int64  
dtypes: float64(3), int64(5), str(7)
memory usage: 685.8 KB


In [152]:
# Fill missing values first
df['fuelType'] = df['fuelType'].fillna('unknown')
df['vehicle.make'] = df['vehicle.make'].fillna('unknown')
df['vehicle.model'] = df['vehicle.model'].fillna('unknown')
df['vehicle.type'] = df['vehicle.type'].fillna('unknown')
df['rating'] = df['rating'].fillna(df['rating'].median())

# Keep readable copies before encoding overwrites them
df['vehicle_type_raw'] = df['vehicle.type']
df['fuelType_raw'] = df['fuelType']
df['vehicle_make_raw'] = df['vehicle.make']

from sklearn.preprocessing import LabelEncoder

# Label encode high-cardinality columns
label_cols = ['vehicle.make', 'vehicle.model']
encoders = {}
for col in label_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# One-hot encode low-cardinality columns
df = pd.get_dummies(df, columns=['fuelType', 'vehicle.type'], prefix=['fuel', 'type'])

In [153]:
#Finds every column that was created by the one-hot encoding
onehot_cols = [c for c in df.columns if c.startswith('fuel_') or c.startswith('type_')]
#Builds the full list of input columns for the model
feature_cols = ['rating', 'renterTripsTaken', 'vehicle.make', 'vehicle.model', 'vehicle.year'] + onehot_cols

X = df[feature_cols]
#model input
y = df['rate.daily']
#target variable

mask = y.notna()
X, y = X[mask], y[mask]
#Applies that mask to both X and y, keeping only the rows where the price isn't missing.

print("NaNs in X:", X.isna().sum().sum())
#counts missing values per column, and the outer .sum()


NaNs in X: 0


In [154]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#split the data into training and testing sets, with 20% of the data reserved for testing.

In [155]:
#Create a copy of the feature dataframe to store results
df_results = X.copy()
df_results['predicted_price'] = model.predict(X)
df_results['actual_rating'] = df.loc[X.index, 'rating']
df_results['vehicle_type_raw'] = df.loc[X.index, 'vehicle_type_raw']
df_results['fuelType_raw'] = df.loc[X.index, 'fuelType_raw']
df_results['vehicle_make_raw'] = df.loc[X.index, 'vehicle_make_raw']

In [156]:
#This creates a new column, value_score, by dividing each car's real rating by its model-predicted price.
df_results['value_score'] = df_results['actual_rating'] / df_results['predicted_price']

In [157]:
# Average rating, price, and value score for each vehicle type/fuel/make combination
grouped = df_results.groupby(['vehicle_type_raw', 'fuelType_raw', 'vehicle_make_raw']).agg(
    avg_rating=('actual_rating', 'mean'),
    avg_predicted_price=('predicted_price', 'mean'),
    avg_value_score=('value_score', 'mean'),
    count=('value_score', 'size')
).reset_index()

In [158]:
#reccomend cars function
def recommend_cars(vehicle_type=None, fuel_type=None, max_budget=None, top_n=5):
    filtered = df_results.copy()

    if vehicle_type:
        filtered = filtered[filtered['vehicle_type_raw'] == vehicle_type]
    if fuel_type:
        filtered = filtered[filtered['fuelType_raw'] == fuel_type]
    if max_budget:
        filtered = filtered[filtered['predicted_price'] <= max_budget]

    return filtered.sort_values('value_score', ascending=False).head(top_n)

In [159]:
# Filter out categories with fewer than 3 observations
grouped = grouped[grouped['count'] >= 3]

# Get the best-performing car in each category
best_per_category = grouped.sort_values('avg_value_score', ascending=False).reset_index(drop=True)
print(best_per_category.head(20))

   vehicle_type_raw fuelType_raw vehicle_make_raw  avg_rating  \
0               car     GASOLINE            smart    4.850769   
1               car     GASOLINE            Scion    4.849231   
2               car     GASOLINE           Toyota    4.893477   
3               car      unknown          Hyundai    4.945000   
4               car      unknown       Mitsubishi    4.750000   
5               car      unknown           Nissan    4.982500   
6               car       HYBRID           Toyota    4.922613   
7               car     GASOLINE          Hyundai    4.904857   
8               car       HYBRID            Honda    4.918095   
9               car     GASOLINE              Kia    4.885408   
10              car     GASOLINE       Volkswagen    4.898152   
11              car     GASOLINE            Honda    4.913115   
12              car     GASOLINE           Nissan    4.898976   
13              car     GASOLINE         Chrysler    4.917241   
14              car     G

In [160]:
#actual evaluation of the model's performance on the test set
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor

param_distributions = {
    'n_estimators': [200, 300, 400, 500, 600, 800, 1000],
    'max_depth': [2, 3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None]
}

In [161]:
#hyperparameter tuning using RandomizedSearchCV to find the best parameters.
gb = GradientBoostingRegressor(random_state=42)

random_search = RandomizedSearchCV(
    gb,
    param_distributions=param_distributions,
    n_iter=50,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
model = random_search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.9, 'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 6, 'learning_rate': 0.05}


In [162]:

# Evaluate the model's predictions on the test set using MAE and R2
from sklearn.metrics import mean_absolute_error, r2_score

preds = model.predict(X_test)    
print('MAE:', mean_absolute_error(y_test, preds))
print('R2:', r2_score(y_test, preds))

MAE: 29.138487182490724
R2: 0.4098144654073831


In [163]:
#If a user's filters are too strict, the result comes back empty. Fallback feature.
def recommend_cars(vehicle_type=None, fuel_type=None, max_budget=None, top_n=5):
    filtered = df_results.copy()

    if vehicle_type:
        filtered = filtered[filtered['vehicle_type_raw'] == vehicle_type]
    if fuel_type:
        filtered = filtered[filtered['fuelType_raw'] == fuel_type]
    if max_budget:
        filtered = filtered[filtered['predicted_price'] <= max_budget]

    if filtered.empty:
        return "No cars match those criteria — try a higher budget or fewer filters."

    return filtered.sort_values('value_score', ascending=False).head(top_n)

In [164]:
#test recommendation feature
recommend_cars(vehicle_type='car', fuel_type='GASOLINE', max_budget=80, top_n=5)

,rating,renterTripsTaken,vehicle.make,vehicle.model,vehicle.year,fuel_DIESEL,fuel_ELECTRIC,fuel_GASOLINE,fuel_HYBRID,fuel_unknown,...,type_minivan,type_suv,type_truck,type_van,predicted_price,actual_rating,vehicle_type_raw,fuelType_raw,vehicle_make_raw,value_score
955,5.00,0,50,510,2008,False,False,True,False,False,...,False,False,False,False,19.920367,5.00,car,GASOLINE,Toyota,0.250999
4791,4.80,18,50,510,2009,False,False,True,False,False,...,False,False,False,False,22.674514,4.80,car,GASOLINE,Toyota,0.211691
1764,4.92,17,50,131,2009,False,False,True,False,False,...,False,False,False,False,23.272399,4.92,car,GASOLINE,Toyota,0.211409
4542,5.00,0,46,525,2009,False,False,True,False,False,...,False,False,False,False,24.175013,5.00,car,GASOLINE,Scion,0.206825
620,5.00,0,19,160,2014,False,False,True,False,False,...,False,False,False,False,25.258488,5.00,car,GASOLINE,Hyundai,0.197953


In [175]:
import joblib
joblib.dump(model, 'model.pkl')
joblib.dump(encoders, 'encoders.pkl')
joblib.dump(list(X.columns), 'feature_order.pkl')

# Save the precomputed results table used by recommend_cars
df_results.to_pickle('df_results.pkl')

In [ ]:
# Label cars as high demand (1) if their trip count is at or above the median, otherwise low demand (0)
median_trips = df['renterTripsTaken'].median()
df['high_demand'] = (df['renterTripsTaken'] >= median_trips).astype(int)

In [ ]:
#create a new dataframe for classification model
feature_cols_clf = ['vehicle.make', 'value_score', 'rating'] + \
                    [c for c in df.columns if c.startswith('fuel_') or c.startswith('type_')]

X_clf = df[feature_cols_clf]
y_clf = df['high_demand']

mask = X_clf.notna().all(axis=1)
X_clf, y_clf = X_clf[mask], y_clf[mask]

In [178]:
from sklearn.model_selection import train_test_split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

In [179]:
#hyperparameter tuning for the classification model using RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier

param_distributions_clf = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8]
}

gbc = GradientBoostingClassifier(random_state=42)

random_search_clf = RandomizedSearchCV(
    gbc,
    param_distributions=param_distributions_clf,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search_clf.fit(X_train_clf, y_train_clf)

print("Best params:", random_search_clf.best_params_)
clf = random_search_clf.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.7, 'n_estimators': 400, 'min_samples_split': 20, 'min_samples_leaf': 8, 'max_depth': 4, 'learning_rate': 0.03}


In [ ]:
#evaluate the classification model's performance on the test set The sort of a scientist are child she rock bearing and now platally we have the inland tighten AKA the fierce snake in one spectacular example of a bit of mistake is this now as you can see even most minimum mistake on Earth, its first instinct is to believe it's really old you can come once it feels like it needs to be you can see it's trying to inch its way this can kill a room of a hundred of people
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

preds_clf = clf.predict(X_test_clf)
print("Accuracy:", accuracy_score(y_test_clf, preds_clf))
print(classification_report(y_test_clf, preds_clf))
print(confusion_matrix(y_test_clf, preds_clf))

Accuracy: 0.7890691716481639
              precision    recall  f1-score   support

           0       0.75      0.86      0.80       571
           1       0.84      0.72      0.78       600

    accuracy                           0.79      1171
   macro avg       0.79      0.79      0.79      1171
weighted avg       0.80      0.79      0.79      1171

[[489  82]
 [165 435]]


In [181]:
import joblib
joblib.dump(clf, 'demand_classifier.pkl')
joblib.dump(feature_cols_clf, 'demand_feature_order.pkl')

['demand_feature_order.pkl']